In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal

In [2]:
class QuadState(TypedDict):

    a: int
    b: int
    c: int

    equation: str
    disc: float
    result: str

In [10]:
def show_eq(state):
    a = state["a"]
    b = state["b"]
    c = state["c"]

    eq = f"{a}x²"

    # Handle b term
    if b > 0:
        eq += f" + {b}x"
    elif b < 0:
        eq += f" - {abs(b)}x"

    # Handle c term
    if c > 0:
        eq += f" + {c}"
    elif c < 0:
        eq += f" - {abs(c)}"

    print("Equation:", eq)
    return {"eq": eq}


def calc_disc(state: QuadState):
    disc = state["b"] ** 2 - (4 * state["a"] * state["c"])
    return {"disc": disc}


def real_roots(state: QuadState):
    root1 = (-state["b"] + state["disc"] ** 0.5) / (2 * state["a"])
    root2 = (-state["b"] - state["disc"] ** 0.5) / (2 * state["a"])

    result = f"the roots are {root1} and {root2}"

    return {"result": result}


def repeated_roots(state: QuadState):
    root = (-state["b"]) / (2 * state["a"])

    result = f"the root is {root}"

    return {"result": result}


def no_real_roots(state: QuadState):

    result = f"No real roots"

    return {"result": result}

In [4]:
def check_condition(state: QuadState) -> Literal['real_roots', 'repeated_roots', 'no_real_roots']:

    if state['disc'] > 0:
        return 'real_roots'
    elif state['disc'] == 0:
        return 'repeated_roots'
    else:
        return 'no_real_roots'         

In [12]:
graph = StateGraph(QuadState)

graph.add_node('show_eq', show_eq)
graph.add_node('calc_disc', calc_disc)
graph.add_node('real_roots', real_roots)
graph.add_node('repeated_roots', repeated_roots)
graph.add_node('no_real_roots', no_real_roots)

graph.add_edge(START, 'show_eq')
graph.add_edge('show_eq', 'calc_disc')
graph.add_conditional_edges('calc_disc', check_condition)
graph.add_edge('real_roots', END)
graph.add_edge('repeated_roots', END)
graph.add_edge('no_real_roots', END)

workflow = graph.compile()

In [13]:
initial_state = {
    'a': 4,
    'b': 16,
    'c': 4
}

final_stage = workflow.invoke(initial_state)
print(final_stage)

Equation: 4x² + 16x + 4
{'a': 4, 'b': 16, 'c': 4, 'disc': 192, 'result': 'the roots are -0.2679491924311228 and -3.732050807568877'}
